In [1]:
import pandas as pd
fp = "../data/sba_loans_prepared/foia-7a-fy2020-present-as-of-251231.csv"
df = pd.read_csv(fp, low_memory=False)

In [2]:
df.head()

,asofdate,program,l2locid,borrname,borrstreet,borrcity,borrstate,borrzip,bankname,bankfdicnumber,...,businesstype,businessage,loanstatus,paidinfulldate,chargeoffdate,grosschargeoffamount,revolverstatus,jobssupported,collateralind,soldsecmrktind
0,12/31/2025,7A,507814.0,Plaza Drive Investments LLC,36223 PLAZA DR,CATHEDRAL CITY,CA,92234,"VelocitySBA, LLC",NaN,...,CORPORATION,Existing or more than 2 years old,EXEMPT,NaN,NaN,0.0,0,3,Y,Y
1,12/31/2025,7A,33850.0,Green Mountain Corporation,19301 S Santa Fe Ave,COMPTON,CA,90221,Comerica Bank,983.0,...,CORPORATION,Existing or more than 2 years old,PIF,10/31/2022,NaN,0.0,1,0,Y,NaN
2,12/31/2025,7A,112407.0,Robert D. Thompson and Lilia A. Garcia,8605 Sovereign Row,Dallas,TX,75247,Enterprise Bank & Trust,27237.0,...,CORPORATION,Existing or more than 2 years old,PIF,3/31/2023,NaN,0.0,0,4,Y,Y
3,12/31/2025,7A,85712.0,FordPowerSolutions LLC,520 SWEETWATER BRIDGE CIR,DOUGLASVILLE,GA,30134,United Midwest Savings Bank National Association,32441.0,...,CORPORATION,"Startup, Loan Funds will Open Business",EXEMPT,NaN,NaN,0.0,0,8,Y,Y
4,12/31/2025,7A,317954.0,Link Rec of Minong Inc.,304 Business Highway 53,MINONG,WI,54859,"Newtek Small Business Finance, Inc.",NaN,...,CORPORATION,Existing or more than 2 years old,PIF,4/30/2024,NaN,0.0,0,12,Y,Y


In [3]:
N = df.shape[0]
cols  = df.columns.tolist()
print(f"Number of records: {N}")
nan_counts = {col: df[col].isna().sum().item() for col in cols}
df_dq_summ = pd.DataFrame.from_dict(nan_counts, orient='index', columns=['nan_count']) 
df_dq_summ['total_count'] = N
df_dq_summ['nan_percentage'] = df_dq_summ['nan_count'] / N * 100
df_dq_summ['nan_percentage'] = df_dq_summ['nan_percentage'].round(2)
df_dq_summ.sort_values(by='nan_percentage', ascending=False).head(10)

Number of records: 357866


,nan_count,total_count,nan_percentage
chargeoffdate,353006,357866,98.64
bankncuanumber,347811,357866,97.19
franchisename,315706,357866,88.22
franchisecode,315557,357866,88.18
paidinfulldate,301049,357866,84.12
soldsecmrktind,251402,357866,70.25
firstdisbursementdate,65730,357866,18.37
bankfdicnumber,37964,357866,10.61
naicsdescription,26998,357866,7.54
bankstreet,498,357866,0.14


In [4]:
df.loanstatus.unique()

<ArrowStringArray>
['EXEMPT', 'PIF', 'CHGOFF', 'CANCLD', 'COMMIT']
Length: 5, dtype: str

In [5]:
sel_charged_off = df['loanstatus'] == 'CHGOFF'
df_charged_off = df[sel_charged_off]
N_charged_off = df_charged_off.shape[0]
print(f"Number of charged off loans: {N_charged_off}")
df_charged_off.head()

Number of charged off loans: 4865


,asofdate,program,l2locid,borrname,borrstreet,borrcity,borrstate,borrzip,bankname,bankfdicnumber,...,businesstype,businessage,loanstatus,paidinfulldate,chargeoffdate,grosschargeoffamount,revolverstatus,jobssupported,collateralind,soldsecmrktind
5,12/31/2025,7A,53803.0,Chaiya Thai Corporation,272 CLAREMONT BLVD,SAN FRANCISCO,CA,94127,"U.S. Bank, National Association",6548.0,...,CORPORATION,Existing or more than 2 years old,CHGOFF,NaN,1/30/2025,3162.07,0,2,Y,NaN
8,12/31/2025,7A,48270.0,Flypixe LLC,34116 CHAGRIN BLVD Apt 9105,CHAGRIN FALLS,OH,44022,"JPMorgan Chase Bank, National Association",628.0,...,CORPORATION,Existing or more than 2 years old,CHGOFF,NaN,4/2/2025,42368.47,1,0,Y,NaN
86,12/31/2025,7A,29805.0,FORTUNE IMPORT & EXPORT INC,311 Bay 10th street FLoor 2,Brooklyn,NY,11228,"TD Bank, National Association",18409.0,...,CORPORATION,Unanswered,CHGOFF,NaN,12/14/2023,22111.14,0,0,N,NaN
90,12/31/2025,7A,12096.0,Caffey Transportation LLC,807 IDLEWOOD DR,BAYTOWN,TX,77520,Wells Fargo Bank National Association,3511.0,...,CORPORATION,NaN,CHGOFF,NaN,3/21/2024,4852.20,1,0,N,NaN
107,12/31/2025,7A,124369.0,Richard French,1875 Diesel Drive #9,SACRAMENTO,CA,95838,Five Star Bank,35361.0,...,INDIVIDUAL,Existing or more than 2 years old,CHGOFF,NaN,11/21/2024,151797.12,0,6,Y,Y


In [6]:
exclude_loanstatus = ['EXEMPT', 'CANCLD', "COMMIT"]
sel_exclude = df['loanstatus'].isin(exclude_loanstatus)
df_inculde = df[~sel_exclude]
df_inculde.loanstatus.unique()

<ArrowStringArray>
['PIF', 'CHGOFF']
Length: 2, dtype: str

In [7]:
df_exclude = df[sel_exclude]
df_exclude.shape

(296185, 43)

In [8]:
pif_select = df["loanstatus"] == "PIF"
df_pif = df[pif_select]
N_pif = df_pif.shape[0]
print(f"Number of PIF loans: {N_pif}")
df_pif.shape

Number of PIF loans: 56816


(56816, 43)

In [9]:
imbalance_ratio = N_charged_off / N_pif
print(f"Imbalance ratio (CHGOFF / PIF): {imbalance_ratio:.4f}")

Imbalance ratio (CHGOFF / PIF): 0.0856


In [10]:
## attribute type assignment

cols = df_charged_off.columns.tolist()
for col in cols:
    if col == "loanamount":
        df[col] = pd.to_numeric(df[col], errors='coerce')
    elif "date" in col.lower():
        df[col] = pd.to_datetime(df[col], errors='coerce')
    else:
        df[col] = df[col].astype('category')


In [11]:
dtype_dict = df.dtypes

In [12]:

dtype_dict = dtype_dict.to_dict()

In [13]:
df_include = df_inculde.astype(dtype_dict)
df_charged_off = df_charged_off.astype(dtype_dict)
df_pif = df_pif.astype(dtype_dict)

In [14]:
df_clean = pd.concat([df_include, df_charged_off, df_pif], ignore_index=True)
del df

In [15]:
df_clean

,asofdate,program,l2locid,borrname,borrstreet,borrcity,borrstate,borrzip,bankname,bankfdicnumber,...,businesstype,businessage,loanstatus,paidinfulldate,chargeoffdate,grosschargeoffamount,revolverstatus,jobssupported,collateralind,soldsecmrktind
0,2025-12-31,7A,33850.0,Green Mountain Corporation,19301 S Santa Fe Ave,COMPTON,CA,90221,Comerica Bank,983.0,...,CORPORATION,Existing or more than 2 years old,PIF,2022-10-31,NaT,0.00,1,0,Y,NaN
1,2025-12-31,7A,112407.0,Robert D. Thompson and Lilia A. Garcia,8605 Sovereign Row,Dallas,TX,75247,Enterprise Bank & Trust,27237.0,...,CORPORATION,Existing or more than 2 years old,PIF,2023-03-31,NaT,0.00,0,4,Y,Y
2,2025-12-31,7A,317954.0,Link Rec of Minong Inc.,304 Business Highway 53,MINONG,WI,54859,"Newtek Small Business Finance, Inc.",NaN,...,CORPORATION,Existing or more than 2 years old,PIF,2024-04-30,NaT,0.00,0,12,Y,Y
3,2025-12-31,7A,53803.0,Chaiya Thai Corporation,272 CLAREMONT BLVD,SAN FRANCISCO,CA,94127,"U.S. Bank, National Association",6548.0,...,CORPORATION,Existing or more than 2 years old,CHGOFF,NaT,2025-01-30,3162.07,0,2,Y,NaN
4,2025-12-31,7A,48270.0,Flypixe LLC,34116 CHAGRIN BLVD Apt 9105,CHAGRIN FALLS,OH,44022,"JPMorgan Chase Bank, National Association",628.0,...,CORPORATION,Existing or more than 2 years old,CHGOFF,NaT,2025-04-02,42368.47,1,0,Y,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123357,2025-12-31,7A,29805.0,MBD LANDSCAPE INC,314 Clark St,North Andover,MA,1845,"TD Bank, National Association",18409.0,...,CORPORATION,Existing or more than 2 years old,PIF,2023-06-30,NaT,0.00,1,33,N,NaN
123358,2025-12-31,7A,57328.0,Odds & Ends Events LLC,313 N Evans St,TECUMSEH,MI,49286,The Huntington National Bank,6560.0,...,CORPORATION,New Business or 2 years or less,PIF,2023-10-31,NaT,0.00,0,16,Y,NaN
123359,2025-12-31,7A,26787.0,Mcgar7 Transport LLC,7706 shepherdsville rd,ELIZABETHTOWN,KY,42701,Wilson & Muir Bank & Trust Company,17040.0,...,CORPORATION,"Startup, Loan Funds will Open Business",PIF,2025-07-31,NaT,0.00,0,2,Y,NaN
123360,2025-12-31,7A,59129.0,Bodyworx Physical Therapy PLLC,"8809 S. Sooner Rd, Ste E",Oklahoma City,OK,73135,First Security Bank and Trust Company,17001.0,...,CORPORATION,Existing or more than 2 years old,PIF,2022-07-31,NaT,0.00,0,16,Y,NaN


In [16]:
date_cols = [col for col in df_clean.columns if "date" in col.lower()]

In [17]:
df_clean[date_cols]= df_clean[date_cols].fillna(pd.Timestamp("1900-01-01"))

In [18]:
cat_cols = df_clean.select_dtypes(include='category').columns.tolist()

In [19]:
# Add "NA" category to each categorical column and fill NaN values with "NA"
for col in cat_cols:
    df_clean[col] = df_clean[col].cat.add_categories(["NA"])
    df_clean[col] = df_clean[col].fillna("NA")

In [20]:
sel_pif = df_clean["loanstatus"] == "PIF"
df_pif = df_clean[sel_pif]
N_pif = df_pif.shape[0]
sel_charged_off = df_clean['loanstatus'] == 'CHGOFF'
df_charged_off = df_clean[sel_charged_off]
N_charged_off = df_charged_off.shape[0]
imbalance_ratio = N_charged_off / N_pif
print(f"Imbalance ratio (CHGOFF / PIF) after cleaning: {imbalance_ratio:.4f}")

Imbalance ratio (CHGOFF / PIF) after cleaning: 0.0856


In [21]:
cols = df_clean.columns.tolist()
cols.remove("l2locid")
df_clean = df_clean[cols]

In [22]:
cat_cols = df_clean.select_dtypes(include='category').columns.tolist()

In [23]:
bank_geo_cols = [ "bankcity", "bankstate", "bankstreet", "bankzip"]
borr_geo_cols = [ "borrstreet", "borrcity", "borrstate", "borrzip"]
numeric_cols = [" initialinterestrate", "terminmonths", "grossapproval","sbaguaranteedapproval", "businessage", "grosschargeoffamount"]
cat_cols = [col for col in cat_cols if col not in bank_geo_cols + borr_geo_cols + numeric_cols]

In [24]:
df_clean["loanstatus"].value_counts()

loanstatus
PIF       113632
CHGOFF      9730
CANCLD         0
COMMIT         0
EXEMPT         0
NA             0
Name: count, dtype: int64